<a href="https://colab.research.google.com/github/e23258-lgtm/Statistical-Learning-e23258/blob/main/Copy_of_Bayesian_Inference_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


In [1]:
import numpy as np
import plotly.graph_objects as go

theta = np.linspace(-4,4,500)

def logistic(theta,a,b):
    return 1/(1+np.exp(-a*(theta-b)))

fig = go.Figure()

# One discrimination value with three difficulties
a = 1.2
for b in [-1,0,1]:
    fig.add_trace(
        go.Scatter(
            x=theta,
            y=logistic(theta,a,b),
            mode='lines',
            name=f"a={a}, b={b}"
        )
    )

# Different discrimination
fig.add_trace(
    go.Scatter(
        x=theta,
        y=logistic(theta,2.2,0),
        mode='lines',
        name="a=2.2, b=0"
    )
)

fig.update_layout(
    title="2PL Item Characteristic Curves",
    xaxis_title="Ability (θ)",
    yaxis_title="P(Correct)",
    template="plotly_white"
)

fig.show()

1. The probability that a user correctly answers item \(i\) under the two-parameter logistic (2PL) model is

$$[
P(Y_i=1\mid\Theta=\theta)
=
p_i(\theta)
=
\frac{1}{1+\exp\left[-a_i(\theta-b_i)\right]}.
$$

The discrimination parameter \(a_i\) determines the steepness of the item characteristic curve, whereas the difficulty parameter \(b_i\) determines its horizontal location.

The probability of a correct response equals \(0.5\) when

$$[
\theta=b_i.]
$$

Therefore,

- increasing $$(b_i)$$ shifts the curve to the right,
- decreasing $$(b_i)$$ shifts the curve to the left,
- changing $$(a_i)$$ changes only the steepness of the curve while leaving the midpoint unchanged.

Thus, items with larger values of $$(b_i)$$ require higher latent ability levels before the probability of answering correctly reaches a given value.

2.For a single observed response $$(y_k)$$, where

$$[
y_k\in\{0,1\},
]$$

the likelihood contribution is

$$[
L(y_k\mid\theta)
=
p_k(\theta)^{y_k}
\left[1-p_k(\theta)\right]^{1-y_k},
]$$

where

$$[
p_k(\theta)
=
\frac{1}{1+\exp\left[-a_k(\theta-b_k)\right]}.
]$$

Assuming conditional independence of the responses given the latent ability, the joint likelihood of the running response history

$$[
\mathbf{y}^{(k)}
=
(y_1,y_2,\ldots,y_k)
]$$

is

$$[
L(\mathbf{y}^{(k)}\mid\theta)
=
\prod_{i=1}^{k}
p_i(\theta)^{y_i}
\left[1-p_i(\theta)\right]^{1-y_i}.
]$$

Each newly observed response contributes one additional Bernoulli likelihood term to the product.

3.Initially, the latent ability follows the prior distribution

$$[
f_{\Theta}^{(0)}(\theta)
=
\frac{1}{\sqrt{2\pi}}
\exp\left(-\frac{\theta^2}{2}\right).
]$$

Suppose that after observing the first $$(k-1)$$ responses, the posterior density is

$$[
f_{\Theta\mid\mathbf{Y}^{(k-1)}}
(\theta\mid\mathbf{y}^{(k-1)}).
]$$

When the next response $$(y_k)$$ is observed, Bayes' theorem updates the posterior recursively as

$$[
f_{\Theta\mid\mathbf{Y}^{(k)}}
(\theta\mid\mathbf{y}^{(k)})
\propto
L(y_k\mid\theta)
f_{\Theta\mid\mathbf{Y}^{(k-1)}}
(\theta\mid\mathbf{y}^{(k-1)}).
]$$

Substituting the Bernoulli likelihood gives

$$[
f_{\Theta\mid\mathbf{Y}^{(k)}}
(\theta\mid\mathbf{y}^{(k)})
\propto
p_k(\theta)^{y_k}
\left[1-p_k(\theta)\right]^{1-y_k}
f_{\Theta\mid\mathbf{Y}^{(k-1)}}
(\theta\mid\mathbf{y}^{(k-1)}).
]$$

The normalized posterior density is therefore

$$[
f_{\Theta\mid\mathbf{Y}^{(k)}}
(\theta\mid\mathbf{y}^{(k)})
=
\frac{
L(y_k\mid\theta)
f_{\Theta\mid\mathbf{Y}^{(k-1)}}
(\theta\mid\mathbf{y}^{(k-1)})
}
{
\displaystyle
\int
L(y_k\mid\theta)
f_{\Theta\mid\mathbf{Y}^{(k-1)}}
(\theta\mid\mathbf{y}^{(k-1)})
\,d\theta
}.
]$$


4. Suppose the user correctly answers a highly difficult item, that is,

$$[
y_k=1,
]$$

where the difficulty parameter $$(b_k)$$ is large.

The likelihood function becomes

$$[
L(y_k=1\mid\theta)
=
p_k(\theta)
=
\frac{1}
{1+\exp[-a_k(\theta-b_k)]}.
]$$

For large values of $$(b_k)$$, the probability of a correct response is very small for low values of $$(\theta)$$ and increases rapidly only when $$(\theta)$$ approaches or exceeds $$(b_k)$$.

The updated posterior distribution is

$$[
f_k(\theta)
\propto
p_k(\theta)
f_{k-1}(\theta).
]$$

Consequently, lower values of the latent ability receive smaller posterior weights, while larger ability values receive greater posterior weights. As a result, the peak (mode) of the posterior distribution shifts toward higher values of $$(\theta)$$.

Therefore, answering a highly difficult item correctly provides strong evidence that the user's latent ability is higher than previously believed.


5. The derivative of the logistic response function is

$$[
\frac{dp_k(\theta)}{d\theta}
=
a_k
p_k(\theta)
\left[1-p_k(\theta)\right].
]$$

The discrimination parameter $$(a_k)$$ determines how rapidly the probability of a correct response changes with respect to the latent ability.

When \(a_k\) is large, the item characteristic curve becomes steeper, producing a more informative likelihood function. Consequently, the posterior distribution becomes narrower, indicating a reduction in posterior variance and greater certainty about the user's ability.

Mathematically,

$$[
\operatorname{Var}(\Theta\mid\mathbf{Y})
\downarrow
]$$

as the discrimination parameter increases.

Conversely, when $$(a_k)$$ is small, the likelihood function is relatively flat and contributes little new information. The posterior distribution changes only slightly and remains comparatively broad, reflecting greater uncertainty in the estimated ability.

Hence,

- large $$(a_k)$$ leads to a sharper posterior distribution and faster learning,
- small $$(a_k)$$ leads to a broader posterior distribution and slower learning.


6. A numerical approximation of the posterior distribution can be maintained by evaluating it on a fixed grid of ability values,

$$[
\theta_1,\theta_2,\ldots,\theta_m.
]$$

Initially, the prior distribution is evaluated on the grid,

$$[
f^{(0)}(\theta_j)
=
\frac{1}{\sqrt{2\pi}}
\exp\left(-\frac{\theta_j^2}{2}\right),
]$$

and normalized so that

$$[
\sum_{j=1}^{m}
f^{(0)}(\theta_j)\Delta\theta
=
1.
]$$

After observing response \(y_k\), the likelihood is computed at every grid point,

$$[
L(y_k\mid\theta_j)
=
p_k(\theta_j)^{y_k}
\left[1-p_k(\theta_j)\right]^{1-y_k}.
]$$

The posterior is then updated by multiplying the previous posterior by the likelihood,

$$[
\tilde{f}^{(k)}(\theta_j)
=
L(y_k\mid\theta_j)
f^{(k-1)}(\theta_j).
]$$

Finally, the updated posterior is normalized by computing

$$[
C
=
\sum_{j=1}^{m}
\tilde{f}^{(k)}(\theta_j)\Delta\theta,
]$$

and dividing each grid value by this constant,

$$[
f^{(k)}(\theta_j)
=
\frac{\tilde{f}^{(k)}(\theta_j)}{C}.
]$$

This normalization ensures that the posterior density integrates to one after every sequential update and allows the posterior distribution to be maintained efficiently as each new response is observed.

In [2]:
import numpy as np

theta_grid = np.linspace(-4, 4, 801)

# Standard normal prior
posterior = np.exp(-0.5 * theta_grid**2)
posterior /= np.trapz(posterior, theta_grid)

# Example item
a = 1.5
b = 0.8
y = 1

# Compute response probability
p = 1 / (1 + np.exp(-a * (theta_grid - b)))

# Likelihood
likelihood = p**y * (1 - p)**(1 - y)

# Bayesian update
posterior *= likelihood

# Normalize
posterior /= np.trapz(posterior, theta_grid)

/tmp/ipykernel_2230/600918630.py:7: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_2230/600918630.py:24: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.



7. Initially, both the posterior mean and the maximum a posteriori (MAP) estimate are close to the prior mean of zero because no response information has yet been incorporated. As additional responses are observed, Bayes' theorem updates the posterior distribution sequentially, causing both estimators to move toward the user's true latent ability,

$$[
\theta_{\text{true}}=0.75.
]$$

Although random variation in the simulated responses may produce temporary fluctuations, the estimation error,

$$[
\left|
\widehat{\theta}^{(k)}
-
\theta_{\text{true}}
\right|,
]$$

generally decreases as the number of observed items increases.

At the same time, the posterior distribution becomes progressively narrower, indicating a reduction in posterior variance and increasing confidence in the estimated ability. Consequently, later observations produce relatively small changes in both the posterior mean and the MAP estimate because the accumulated evidence dominates the influence of any single response.

Therefore, as $$(k)$$ increases, both estimators converge toward the true latent ability while the posterior distribution becomes more concentrated, reflecting the platform's increasing confidence in its measurement of the user's ability.

In [3]:
import numpy as np
import plotly.graph_objects as go

np.random.seed(123)

# True ability
theta_true = 0.75

# Number of items
n = 20

# Ability grid
theta_grid = np.linspace(-4, 4, 801)

# Prior distribution
posterior = np.exp(-0.5 * theta_grid**2)
posterior /= np.trapz(posterior, theta_grid)

posterior_mean = [0]
posterior_map = [0]

# Generate random item parameters
a = np.random.uniform(0.5, 2.0, n)
b = np.random.normal(0, 1, n)

def logistic(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Sequential Bayesian updating
for k in range(n):

    # True probability of correct response
    p_true = logistic(theta_true, a[k], b[k])

    # Simulate response
    y = np.random.rand() < p_true

    # Likelihood over the grid
    p_grid = logistic(theta_grid, a[k], b[k])
    likelihood = p_grid if y else (1 - p_grid)

    # Update posterior
    posterior *= likelihood
    posterior /= np.trapz(posterior, theta_grid)

    # Posterior mean
    mean = np.trapz(theta_grid * posterior, theta_grid)

    # MAP estimate
    map_est = theta_grid[np.argmax(posterior)]

    posterior_mean.append(mean)
    posterior_map.append(map_est)

# Plot results
steps = np.arange(n + 1)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_mean,
        mode="lines+markers",
        name="Posterior Mean"
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_map,
        mode="lines+markers",
        name="MAP Estimate"
    )
)

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True Ability = 0.75"
)

fig.update_layout(
    title="Sequential Bayesian Ability Estimation",
    xaxis_title="Item Number",
    yaxis_title="Ability Estimate",
    template="plotly_white"
)

fig.show()

/tmp/ipykernel_2230/3417640712.py:17: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_2230/3417640712.py:44: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_2230/3417640712.py:47: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.



# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

**Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).


**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

1. The probability density function (PDF) of a Beta distribution is

$$[
f(\theta)
=
\frac{1}{\mathrm{B}(\alpha,\beta)}
\theta^{\alpha-1}
(1-\theta)^{\beta-1},
\qquad
0\le\theta\le1,
]$$

where

$$[
\mathrm{B}(\alpha,\beta)
=
\frac{\Gamma(\alpha)\Gamma(\beta)}
{\Gamma(\alpha+\beta)}
]$$

is the Beta function that normalizes the density.

Three different parameter settings illustrate the effect of the shape parameters:

- **Uninformative prior:** $$(\alpha=1,\;\beta=1)$$
- **Right-skewed distribution:** $$(\alpha=2,\;\beta=8)$$
- **Left-skewed distribution:** $$(\alpha=8,\;\beta=2)$$

The mean of a Beta distribution is

$$[
E[\Theta]
=
\frac{\alpha}{\alpha+\beta}.
]$$

Hence,

- when $$(\alpha=\beta)$$, the density is symmetric;
- when $$(\alpha<\beta)$$, most probability mass is concentrated near zero;
- when $$(\alpha>\beta)$$, most probability mass is concentrated near one.

Increasing the value of $$(\alpha)$$ shifts the center of mass toward larger values of $$(\theta)$$, whereas increasing $$(\beta)$$ shifts the center of mass toward smaller values of $$(\theta)$$.

In [4]:
import numpy as np
from scipy.stats import beta
import plotly.graph_objects as go

theta = np.linspace(0, 1, 500)

fig = go.Figure()

parameters = [
    (1,1,"Beta(1,1)"),
    (2,8,"Beta(2,8)"),
    (8,2,"Beta(8,2)")
]

for a,b,label in parameters:
    fig.add_trace(
        go.Scatter(
            x=theta,
            y=beta.pdf(theta,a,b),
            mode="lines",
            name=label
        )
    )

fig.update_layout(
    title="Beta Distribution PDFs",
    xaxis_title="θ",
    yaxis_title="Density",
    template="plotly_white"
)

fig.show()

2. Each user interaction is modeled as a Bernoulli random variable,

$$[
Y_k
\sim
\text{Bernoulli}(\theta),
]$$

where

$$[
P(Y_k=1\mid\Theta=\theta)=\theta.
]$$

For a single observed response,

$$[
y_k\in\{0,1\},
]$$

the likelihood contribution is

$$[
L(y_k\mid\theta)
=
\theta^{y_k}
(1-\theta)^{1-y_k}.
]$$

Assuming conditional independence of user interactions given the click-through rate, the likelihood for the running history

$$[
\mathbf{y}^{(k)}
=
(y_1,y_2,\ldots,y_k)
]$$

is

$$[
L(\mathbf{y}^{(k)}\mid\theta)
=
\prod_{i=1}^{k}
\theta^{y_i}
(1-\theta)^{1-y_i}.
]$$

Combining the exponents gives

$$[
L(\mathbf{y}^{(k)}\mid\theta)
=
\theta^{\sum_{i=1}^{k}y_i}
(1-\theta)^{k-\sum_{i=1}^{k}y_i}.
]$$

Thus, the likelihood depends only on the total number of clicks and non-clicks observed up to step $$(k)$$.

3.The prior distribution at step \(k-1\) is

$$[
\Theta
\mid
\mathbf{Y}^{(k-1)}
\sim
\text{Beta}
(\alpha_{k-1},\beta_{k-1}),
]$$

with density

$$[
f(\theta)
=
\frac{1}
{\mathrm{B}
(\alpha_{k-1},\beta_{k-1})}
\theta^{\alpha_{k-1}-1}
(1-\theta)^{\beta_{k-1}-1}.
]$$

Using Bayes' theorem,

$$[
f(\theta\mid\mathbf{y}^{(k)})
\propto
L(y_k\mid\theta)
f(\theta\mid\mathbf{y}^{(k-1)}).
]$$

Substituting the likelihood,

$$[
L(y_k\mid\theta)
=
\theta^{y_k}
(1-\theta)^{1-y_k},
]$$

gives

$$[
f(\theta\mid\mathbf{y}^{(k)})
\propto
\theta^{y_k}
(1-\theta)^{1-y_k}
\theta^{\alpha_{k-1}-1}
(1-\theta)^{\beta_{k-1}-1}.
]$$

Combining powers,

$$[
f(\theta\mid\mathbf{y}^{(k)})
\propto
\theta^{\alpha_{k-1}+y_k-1}
(1-\theta)^{\beta_{k-1}+1-y_k-1}.
]$$

Therefore,

$$[
\Theta
\mid
\mathbf{Y}^{(k)}
\sim
\text{Beta}
(\alpha_k,\beta_k),
]$$

where

$$[
\boxed{
\alpha_k
=
\alpha_{k-1}+y_k
}
]$$

and

$$[
\boxed{
\beta_k
=
\beta_{k-1}
+
(1-y_k)
}
]$$

These equations show that

- every click increases the parameter $$(\alpha)$$ by one;
- every non-click increases the parameter $$(\beta)$$ by one.

Hence, the posterior remains in the Beta family after every update, demonstrating Beta-Bernoulli conjugacy.

The posterior density is therefore

$$[
f(\theta\mid\mathbf{Y}^{(k)})
=
\frac
{1}
{\mathrm{B}(\alpha_k,\beta_k)}
\theta^{\alpha_k-1}
(1-\theta)^{\beta_k-1}.
]$$

The posterior mean is

$$[
\boxed{
E[\Theta\mid\mathbf{Y}^{(k)}]
=
\frac{\alpha_k}
{\alpha_k+\beta_k}
}
]$$

which represents the Bayesian estimate of the advertisement's click-through rate after observing $$(k)$$ impressions.


4.Suppose the posterior distribution after observing the first \(k-1\) impressions is

$$[
\Theta
\mid
\mathbf{Y}^{(k-1)}
\sim
\mathrm{Beta}(\alpha_{k-1},\beta_{k-1}).
]$$

When a new user interaction is observed, the posterior parameters are updated analytically.

If the user clicks the advertisement,

$$[
y_k=1,
]$$

then

$$[
\alpha_k=\alpha_{k-1}+1,
\qquad
\beta_k=\beta_{k-1}.
]$$

The additional click increases the exponent of $$(\theta)$$, shifting the posterior density toward larger values of the click-through rate. Consequently, the peak of the posterior distribution moves to the right, indicating increased belief that the advertisement has a higher conversion rate.

Conversely, if the user does not click the advertisement,

$$[
y_k=0,
]$$

then

$$[
\alpha_k=\alpha_{k-1},
\qquad
\beta_k=\beta_{k-1}+1.
]$$

The additional non-click increases the exponent of $$((1-\theta))$$, shifting the posterior density toward smaller values of $$(\theta)$$. As a result, the peak of the posterior distribution moves to the left, indicating a lower estimated click-through rate.

This analytical update is possible because the Beta prior is conjugate to the Bernoulli likelihood. The posterior distribution remains a Beta distribution after every observation, requiring only simple arithmetic updates of the shape parameters.

In contrast, non-conjugate Bayesian models such as the two-parameter logistic (2PL) Item Response Theory model do not admit closed-form posterior distributions. Each new observation changes the posterior into a distribution that cannot be expressed analytically. Consequently, numerical approximation methods such as grid approximation, numerical integration, Laplace approximation, or Markov Chain Monte Carlo (MCMC) sampling are required to compute posterior summaries.


5.After observing \(k\) impressions, the posterior distribution is

$$[
\Theta
\mid
\mathbf{Y}^{(k)}
\sim
\mathrm{Beta}(\alpha_k,\beta_k).
]$$

The running Bayesian posterior mean is

$$[
\boxed{
\widehat{\theta}_{\mathrm{Bayes}}^{(k)}
=
\frac{\alpha_k}
{\alpha_k+\beta_k}
}
]$$

which minimizes the posterior mean squared error.

The running Maximum A Posteriori (MAP) estimate is the mode of the Beta distribution and is given by

$$[
\boxed{
\widehat{\theta}_{\mathrm{MAP}}^{(k)}
=
\frac{\alpha_k-1}
{\alpha_k+\beta_k-2},
\qquad
\alpha_k>1,\;
\beta_k>1
}
]$$

When either

$$[
\alpha_k\le1
\quad\text{or}\quad
\beta_k\le1,
]$$

the posterior mode occurs at one of the boundaries,

$$[
\widehat{\theta}_{\mathrm{MAP}}
=
0
\quad\text{or}\quad
1.
]$$

As additional impressions are observed, both estimators gradually converge toward the true click-through rate while the influence of the initial prior decreases.

In [1]:
import numpy as np
import plotly.graph_objects as go

# ----------------------------
# Parameters
# ----------------------------

np.random.seed(123)

theta_true = 0.35
n = 100

# Initial Beta prior
alpha = 1
beta = 1

steps = [0]

posterior_mean = [alpha/(alpha+beta)]

# MAP estimate at step 0
posterior_map = [0.5]

# ----------------------------
# Sequential Bayesian Updating
# ----------------------------

for k in range(1, n+1):

    # Simulate one user response
    y = 1 if np.random.rand() < theta_true else 0

    # Analytical update
    alpha += y
    beta += (1-y)

    # Posterior Mean
    mean = alpha/(alpha+beta)

    # Posterior MAP
    if alpha > 1 and beta > 1:
        map_est = (alpha-1)/(alpha+beta-2)
    elif alpha <= 1:
        map_est = 0
    else:
        map_est = 1

    steps.append(k)
    posterior_mean.append(mean)
    posterior_map.append(map_est)

# ----------------------------
# Plot
# ----------------------------

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_mean,
        mode="lines+markers",
        name="Posterior Mean"
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_map,
        mode="lines+markers",
        name="MAP Estimate"
    )
)

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True CTR = 0.35"
)

fig.update_layout(
    title="Sequential Bayesian Estimation of Advertisement CTR",
    xaxis_title="Impression Number",
    yaxis_title="Estimated Click-Through Rate",
    template="plotly_white"
)

fig.show()

6. Initially, both the posterior mean and the MAP estimate are strongly influenced by the uniform prior,

$$[
\Theta \sim \mathrm{Beta}(1,1),
]$$

which assigns equal probability to every value of the click-through rate between 0 and 1. Consequently, the initial estimates may differ substantially from the true click-through rate,

$$[
\theta_{\mathrm{true}}=0.35.
]$$

As additional user interactions are observed, the parameters of the Beta distribution are updated analytically according to

$$[
\alpha_k=\alpha_{k-1}+y_k,
]$$

and

$$[
\beta_k=\beta_{k-1}+(1-y_k).
]$$
Each click increases the evidence supporting larger values of the click-through rate, whereas each non-click increases the evidence supporting smaller values.

As the number of impressions approaches

$$[
k=100,
]$$

the influence of the initial prior becomes negligible compared with the accumulated observations. Consequently, both the posterior mean and the MAP estimate move progressively closer to the true click-through rate. Although random sampling variability may produce small fluctuations throughout the simulation, the estimation error generally decreases as more data become available.

This behavior demonstrates a fundamental property of Bayesian learning: the posterior distribution becomes increasingly dominated by the observed data, while the influence of the prior diminishes over time. Therefore, the sequential estimators become more stable and provide increasingly accurate estimates of the advertisement's true click-through rate as evidence accumulates.

# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In aerospace and civil engineering, Structural Health Monitoring (SHM) is critical for detecting damage before a catastrophic failure occurs. Consider an aircraft wing or a bridge girder equipped with specialized vibration sensors. Over time, environmental fatigue or dynamic impacts can cause micro-fractures, resulting in a reduction of the component's mechanical stiffness.

Let $\Theta = \theta$ represent the structural **remaining stiffness efficiency factor**, where $\theta$ is physically bounded to the interval:

$$\theta \in (0, 1]$$

* $\theta = 1.0$ indicates a perfectly pristine, undamaged structural component.
* $\theta \to 0$ signifies critical degradation or severe structural cracking.

Let $K_{\text{nominal}}$ be the known, baseline stiffness of the structural component when it is entirely healthy. At each sequential inspection time step $k$ (where $k = 1, 2, \dots, n$), a sensor collects a noisy experimental stiffness measurement $y_k$.

Engineers model the degradation physics via a non-linear relationship with multiplicative log-normal measurement noise to prevent non-physical negative values:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

where $\sigma$ is the standard deviation of the sensor noise in log-space.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running history vector of observed sensor readings** up to the current inspection milestone. Before deploying the sensors, engineers utilize an initial prior distribution $f_{\Theta}^{(0)}(\theta)$ over the domain $(0, 1]$ based on historical manufacturing specifications. As the sensor stream arrives, the posterior distribution calculated at step $k-1$ serves directly as the prior distribution for step $k$.

---

### **Tasks**

#### **1. Prior Belief Boundaries**

Before data collection begins, engineers assume the component is highly likely to be healthy, modeling this using a bounded Beta distribution as the initial prior: $\Theta \sim \text{Beta}(8, 1.5)$.

* Plot this initial prior density function using Plotly over the restricted physical domain $\theta \in [0.01, 1.0]$.
* Calculate the expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ analytically. Explain why this specific distribution serves as an appropriate initial prior for an engineering component assumed to be healthy.

#### **2. Structural Likelihood Formulation**

Using the change of variables or properties of the log-normal distribution, write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* continuous sensor measurement $y_k$ at inspection step $k$, given the true stiffness factor $\theta$. Following this, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

#### **3. Mathematical Formulation of the Non-Conjugate Grid Update**

Explain why an exact closed-form analytical solution for the posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ does not exist when combining a Beta prior with this log-normal structural likelihood. Write down the recursive relationship for the posterior density at step $k$ up to a proportionality constant.

#### **4. Running Point Estimates**

Because a closed-form formula is unavailable, we must define point estimators through numerical integration. Write down the definite integral equations over the bounded domain $(0, 1]$ required to compute:

* The **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* The **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

#### **5. Algorithmic Grid Approximation and Normalization**

Describe the step-by-step numerical procedure to maintain this distribution on a discrete grid of $\theta$-values. Explicitly state how you would handle the boundary limits computationally and how you would perform the sequential normalization step using the trapezoidal rule after a new sensor reading $y_k$ is observed.

#### **6. Performance Tracking and Degradation Convergence Analysis**

Suppose an impact occurs, and the true, hidden remaining stiffness drops to $\theta_{\text{true}} = 0.68$. Write a Python script using Plotly to simulate an engineered monitoring timeline across $n = 15$ continuous sensor measurements ($K_{\text{nominal}} = 50.0 \text{ kN/mm}$, $\sigma = 0.15$):

* **Simulate Sensor Stream:** Programmatically generate noisy sensor readings $y_k$ by drawing random values from the underlying log-normal physics model centered at $\theta_{\text{true}}$.
* **Track Estimators:** Loop sequentially through each step. At each step, update the unnormalized grid, normalize it via `np.trapezoid`, and compute both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize Curves & Timeline:** Generate two plots:
1. A plot showing the progression of the full posterior density curves at milestones $k \in \{0, 1, 2, 5, 10, 15\}$.
2. A line chart tracking the convergence of both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ from step $0$ to $15$ against a horizontal reference line at $\theta_{\text{true}} = 0.68$.


* **Analysis:** Evaluate the behavior of the distribution. How many sensor readings did it take for the system to overcome the initially optimistic "healthy" prior and confidently isolate the 68% damage state? What does the narrowing of the density curves imply about structural safety thresholds?

1. Before any sensor measurements are collected, the remaining stiffness efficiency factor is assumed to follow the prior distribution

$$[
\Theta
\sim
\mathrm{Beta}(8,1.5),
]$$

with probability density function

$$[
f_{\Theta}^{(0)}(\theta)
=
\frac{1}
{\mathrm{B}(8,1.5)}
\theta^{8-1}
(1-\theta)^{1.5-1},
\qquad
0<\theta\le1.
]$$

The Beta distribution is defined only over the bounded interval

$$[
0<\theta<1,
]$$

which matches the physical constraint that the remaining stiffness efficiency cannot exceed one or become negative.

The expected value of a Beta distribution is

$$[
\mathbb{E}\left[\Theta^{(0)}\right]
=
\frac{\alpha}{\alpha+\beta}.
]$$

Substituting

$$[
\alpha=8,
\qquad
\beta=1.5,
]$$

gives

$$[
\mathbb{E}\left[\Theta^{(0)}\right]
=
\frac{8}{8+1.5}
=
\frac{8}{9.5}
=
0.8421.
]$$

Therefore,

$$[
\boxed{
\mathbb{E}\left[\Theta^{(0)}\right]
\approx0.8421
}
]$$

indicating that engineers initially expect the component to retain approximately $$(84.2\%)$$ of its original stiffness.

This prior is appropriate because newly manufactured engineering components are generally expected to be in good condition before service begins. Consequently, most of the probability mass is concentrated near $$(\theta=1)$$, reflecting high confidence that the structure is initially healthy while still allowing for moderate uncertainty arising from manufacturing tolerances and material variability.

In [2]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

theta = np.linspace(0.01,1.0,500)

prior = beta.pdf(theta,8,1.5)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=theta,
        y=prior,
        mode="lines",
        name="Beta(8,1.5)"
    )
)

fig.update_layout(
    title="Prior Distribution of Remaining Stiffness Efficiency",
    xaxis_title="Remaining Stiffness Efficiency (θ)",
    yaxis_title="Density",
    template="plotly_white"
)

fig.show()

2. The structural degradation model is

$$[
y_k
=
\theta
K_{\mathrm{nominal}}
e^{\epsilon_k},
]$$

where

$$[
\epsilon_k
\sim
\mathcal N(0,\sigma^2).
]$$

Taking the natural logarithm gives

$$[
\ln y_k
=
\ln(\theta K_{\mathrm{nominal}})
+
\epsilon_k.
]$$

Since

$$[
\epsilon_k
\sim
\mathcal N(0,\sigma^2),
]$$

it follows that

$$[
\ln y_k
\sim
\mathcal N
\left(
\ln(\theta K_{\mathrm{nominal}}),
\sigma^2
\right).
]$$

Therefore, the likelihood contribution of a single sensor measurement is the log-normal density

$$[
L(y_k\mid\theta)
=
\frac{1}
{y_k\sigma\sqrt{2\pi}}
\exp
\left[
-
\frac{
\left(
\ln y_k
-
\ln(\theta K_{\mathrm{nominal}})
\right)^2
}
{2\sigma^2}
\right].
]$$

Assuming conditional independence of the measurements given the remaining stiffness factor, the likelihood for the running history vector

$$[
\mathbf y^{(k)}
=
(y_1,y_2,\ldots,y_k)
]$$

is

$$[
L(\mathbf y^{(k)}\mid\theta)
=
\prod_{i=1}^{k}
L(y_i\mid\theta).
]$$

Substituting the log-normal density gives

$$[
L(\mathbf y^{(k)}\mid\theta)
=
\prod_{i=1}^{k}
\frac{1}
{y_i\sigma\sqrt{2\pi}}
\exp
\left[
-
\frac{
\left(
\ln y_i
-
\ln(\theta K_{\mathrm{nominal}})
\right)^2
}
{2\sigma^2}
\right].
]$$

Thus, the likelihood incorporates information from every sensor measurement observed up to inspection step $$(k)$$.

3. The prior distribution is

$$[
\Theta
\sim
\mathrm{Beta}(\alpha,\beta),
]$$

whereas the measurement likelihood follows a log-normal distribution.

Unlike the Beta-Bernoulli model, the Beta distribution is not conjugate to the log-normal likelihood. Multiplying the Beta density by the log-normal likelihood does not produce another probability distribution belonging to a known family.

Consequently, no closed-form analytical expression exists for the posterior distribution, and the normalizing constant cannot be evaluated symbolically. Numerical approximation methods such as grid approximation or Markov Chain Monte Carlo are therefore required.

Using Bayes' theorem, the recursive posterior update is

$$[
f_{\Theta\mid\mathbf Y^{(k)}}
(\theta\mid\mathbf y^{(k)})
\propto
L(y_k\mid\theta)
f_{\Theta\mid\mathbf Y^{(k-1)}}
(\theta\mid\mathbf y^{(k-1)}).
]$$

Substituting the log-normal likelihood,

$$[
f_{\Theta\mid\mathbf Y^{(k)}}
(\theta\mid\mathbf y^{(k)})
\propto
\frac{1}
{y_k\sigma\sqrt{2\pi}}
\exp
\left[
-
\frac{
\left(
\ln y_k
-
\ln(\theta K_{\mathrm{nominal}})
\right)^2
}
{2\sigma^2}
\right]
f_{\Theta\mid\mathbf Y^{(k-1)}}
(\theta\mid\mathbf y^{(k-1)}).
]$$

After computing the unnormalized posterior on a numerical grid, the distribution is normalized so that

$$[
\int_0^1
f_{\Theta\mid\mathbf Y^{(k)}}
(\theta\mid\mathbf y^{(k)})
\,d\theta
=
1.
]$$

This recursive numerical update is repeated after every new sensor measurement, allowing the posterior distribution to evolve sequentially as additional structural health information becomes available.

4. Since the posterior distribution does not have a closed-form analytical expression, point estimates must be computed numerically from the posterior density defined over the bounded domain

$$[
0<\theta\le1.
]$$

The running posterior mean (Bayesian estimator) is obtained by evaluating

$$[
\boxed{
\widehat{\theta}_{\mathrm{Bayes}}^{(k)}
=
\int_{0}^{1}
\theta\,
f_{\Theta|\mathbf{Y}^{(k)}}
(\theta|\mathbf{y}^{(k)})
\,d\theta
}
]$$

which represents the expected remaining stiffness efficiency after incorporating all sensor measurements up to inspection step \(k\).

The Running Maximum A Posteriori (MAP) estimate is the value of the parameter that maximizes the posterior density,

$$[
\boxed{
\widehat{\theta}_{\mathrm{MAP}}^{(k)}
=
\underset{0<\theta\le1}{\operatorname{arg\,max}}
\;
f_{\Theta|\mathbf{Y}^{(k)}}
(\theta|\mathbf{y}^{(k)})
}
]$$

Since the posterior distribution has no analytical form, both estimators must be evaluated numerically.

The posterior mean is approximated by numerical integration over the grid,

$$[
\widehat{\theta}_{\mathrm{Bayes}}^{(k)}
\approx
\sum_{j=1}^{m}
\theta_j
f(\theta_j)
\Delta\theta,
]$$

whereas the MAP estimate is obtained by locating the grid point corresponding to the maximum posterior density,

$$[
\widehat{\theta}_{\mathrm{MAP}}^{(k)}
=
\theta_j
\quad
\text{such that}
\quad
f(\theta_j)
=
\max_i
f(\theta_i).
]$$

 5. Since the posterior distribution cannot be computed analytically, it is approximated numerically on a fixed grid of stiffness efficiency values,

$$[
\theta_1,\theta_2,\ldots,\theta_m,
]$$

covering the physical interval

$$[
0<\theta\le1.
]$$

To avoid evaluating the logarithm at zero,

$$[
\ln(0),
]$$

the grid is initialized from a small positive value, for example,

$$[
\theta\in[0.01,1.00].
]$$

The sequential algorithm proceeds as follows.

1. Construct the initial Beta prior on the grid.

2. Normalize the prior using the trapezoidal rule,

$$[
f(\theta)
\leftarrow
\frac{f(\theta)}
{\displaystyle
\int_{0.01}^{1}
f(\theta)\,d\theta}.
]$$

3. When a new sensor measurement $$(y_k)$$ is observed, evaluate the log-normal likelihood at every grid point,

$$[
L(y_k|\theta_j).
]$$

4. Compute the unnormalized posterior,

$$[
\tilde f^{(k)}(\theta_j)
=
L(y_k|\theta_j)
f^{(k-1)}(\theta_j).
]$$

5. Compute the normalization constant numerically using the trapezoidal rule,

$$[
C
=
\int_{0.01}^{1}
\tilde f^{(k)}(\theta)
\,d\theta.
]$$

6. Normalize the posterior,

$$[
f^{(k)}(\theta_j)
=
\frac
{\tilde f^{(k)}(\theta_j)}
{C}.
]$$

7. Evaluate the posterior mean and MAP estimate from the normalized posterior.

These steps are repeated sequentially after every new sensor measurement, allowing the posterior distribution to evolve continuously as additional structural health information becomes available.

In [3]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# ----------------------------
# Parameters
# ----------------------------

np.random.seed(123)

theta_true = 0.68
K_nominal = 50.0
sigma = 0.15
n = 15

# Grid
theta_grid = np.linspace(0.01, 1.0, 800)

# Prior
posterior = beta.pdf(theta_grid, 8, 1.5)
posterior /= np.trapezoid(posterior, theta_grid)

# Store estimates
posterior_mean = [
    np.trapezoid(theta_grid * posterior, theta_grid)
]

posterior_map = [
    theta_grid[np.argmax(posterior)]
]

steps = [0]

# Save posterior curves
posterior_history = {0: posterior.copy()}

# ----------------------------
# Sequential Updating
# ----------------------------

for k in range(1, n + 1):

    # Simulate noisy sensor measurement
    y = theta_true * K_nominal * np.exp(
        np.random.normal(0, sigma)
    )

    # Log-normal likelihood
    likelihood = (
        1
        / (y * sigma * np.sqrt(2 * np.pi))
        * np.exp(
            -(
                np.log(y)
                - np.log(theta_grid * K_nominal)
            ) ** 2
            / (2 * sigma**2)
        )
    )

    # Bayesian update
    posterior *= likelihood

    # Normalize
    posterior /= np.trapezoid(
        posterior,
        theta_grid
    )

    # Posterior Mean
    mean = np.trapezoid(
        theta_grid * posterior,
        theta_grid
    )

    # MAP
    map_est = theta_grid[
        np.argmax(posterior)
    ]

    steps.append(k)
    posterior_mean.append(mean)
    posterior_map.append(map_est)

    if k in [1, 2, 5, 10, 15]:
        posterior_history[k] = posterior.copy()

# ----------------------------
# Posterior Density Plot
# ----------------------------

fig1 = go.Figure()

for k in [0, 1, 2, 5, 10, 15]:
    fig1.add_trace(
        go.Scatter(
            x=theta_grid,
            y=posterior_history[k],
            mode="lines",
            name=f"k={k}"
        )
    )

fig1.update_layout(
    title="Evolution of Posterior Density",
    xaxis_title="Remaining Stiffness Efficiency (θ)",
    yaxis_title="Posterior Density",
    template="plotly_white"
)

fig1.show()

# ----------------------------
# Estimator Convergence
# ----------------------------

fig2 = go.Figure()

fig2.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_mean,
        mode="lines+markers",
        name="Posterior Mean"
    )
)

fig2.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_map,
        mode="lines+markers",
        name="MAP Estimate"
    )
)

fig2.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True Stiffness = 0.68"
)

fig2.update_layout(
    title="Sequential Estimation of Remaining Stiffness",
    xaxis_title="Inspection Number",
    yaxis_title="Remaining Stiffness Efficiency",
    template="plotly_white"
)

fig2.show()

6. Initially, the posterior distribution is strongly influenced by the optimistic prior

$$[
\Theta
\sim
\mathrm{Beta}(8,1.5),
]$$

which concentrates most of its probability mass near

$$[
\theta=1.
]$$

Consequently, the initial posterior mean and MAP estimate are substantially larger than the true remaining stiffness,

$$[
\theta_{\mathrm{true}}=0.68.
]$$

As sequential sensor measurements are incorporated, each new log-normal likelihood contributes additional evidence regarding the actual structural condition. The posterior distribution gradually shifts toward lower stiffness values, eventually overcoming the optimistic prior assumption.

In a typical simulation, approximately five to ten sensor measurements are sufficient for the posterior distribution to become centered close to the true remaining stiffness. After approximately ten to fifteen observations, both the posterior mean and MAP estimate usually fluctuate only slightly around

$$[
\theta_{\mathrm{true}}=0.68,
]$$

indicating that the accumulated sensor evidence dominates the influence of the initial prior.

The posterior density curves become progressively narrower as more observations are collected. This reduction in posterior spread indicates decreasing uncertainty regarding the structural condition. A narrow posterior distribution implies that the monitoring system has become increasingly confident in its estimate of the remaining stiffness, enabling engineers to make more reliable maintenance and safety decisions.

From a structural health monitoring perspective, the progressive concentration of the posterior density provides quantitative evidence regarding structural safety thresholds. Once the posterior probability becomes concentrated around a degraded stiffness level, maintenance actions or detailed inspections can be initiated with significantly greater confidence than would be possible using only the initial engineering assumptions.

# Q. Gaussian Mixture Clustering as Conditional Updating

Consider a dataset
$$
x_1,x_2,\dots,x_n\in\mathbb R^d.
$$
We wish to cluster these observations into $K$ groups. Instead of assigning each point deterministically to a cluster at the beginning, we introduce a latent random variable
$$
C_i\in{1,\dots,K},
$$
where $C_i=k$ means that the observation $x_i$ belongs to cluster $k$.
Let the prior probability of cluster membership be
$$
P(C_i=k)=\phi_k,
$$
where
$$
\phi_k\ge 0,
\qquad
\sum_{k=1}^K \phi_k=1.
$$

Conditional on $C_i=k$, assume that the observation $X_i$ is generated from a multivariate Gaussian distribution:
$$
X_i\mid C_i=k
\sim
\mathscr N(\mu_k,\Sigma_k),
$$
where
$$
\mu_k\in\mathbb R^d,
\qquad
\Sigma_k\in\mathbb R^{d\times d}
$$
are the mean vector and covariance matrix of cluster $k$.

The model parameters
$$
\phi_k,\mu_k,\Sigma_k,
\qquad k=1,\dots,K,
$$
are assumed to be fixed but unknown.

---

1. Deriving the Marginal Density:
Using the law of total probability, show that the marginal density of $X_i$ is
$$
p(x_i)=\sum_{k=1}^K
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k).
$$
Explain why this density is called a Gaussian mixture density.

---

2. Deriving the Posterior Cluster Probability:
For a fixed observation $x_i$, use Bayes' rule to derive
$$
P(C_i=k\mid X_i=x_i)=\frac{
P(X_i=x_i\mid C_i=k)P(C_i=k)
}{
\sum_{j=1}^K P(X_i=x_i\mid C_i=j)P(C_i=j)
}.
$$
Then substitute the Gaussian model and the cluster prior to obtain
$$
P(C_i=k\mid X_i=x_i)=\frac{
\phi_k\mathscr N(x_i\mid \mu_k,\Sigma_k)
}{
\sum_{j=1}^K
\phi_j\mathscr N(x_i\mid \mu_j,\Sigma_j)
}.
$$
This quantity is called the responsibility of cluster $k$ for data point $x_i$, and is denoted by
$$
\gamma_{ik}=P(C_i=k\mid X_i=x_i).
$$
Explain why $\gamma_{ik}$ may be interpreted as a posterior probability of cluster membership.

---

3. One-Hot Encoding of the Latent Cluster Variable:
Now define a one-hot encoded latent random vector
$$
Z_i=
\begin{bmatrix}
Z_{i1}\\
Z_{i2}\\
\vdots\\
Z_{iK}
\end{bmatrix},
$$
where
$$
Z_{ik}=\begin{cases}
1, & \text{if } C_i=k,\\
0, & \text{otherwise}.
\end{cases}
$$
Show that
$$
\mathbb E[Z_{ik}\mid X_i=x_i]=P(C_i=k\mid X_i=x_i).
$$
Hence show that
$$
\mathbb E[Z_i\mid X_i=x_i]=\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}.
$$
Conclude that the soft cluster assignment in a Gaussian mixture model is precisely the conditional expectation
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

---

4. From Soft Assignment to Hard Clustering:
The vector
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
gives a soft assignment of $x_i$ to all clusters. A hard cluster assignment can be obtained by choosing the cluster with the largest posterior probability:
$$
\widehat C_i=\operatorname{arg\,max}_{1\le k\le K}
\gamma_{ik}.
$$
Explain the difference between soft clustering and hard clustering in this context.

---

5. Conditional Expectation of the Observation Given the Cluster:
Show that
$$
\mathbb E[X_i\mid C_i=k]=\mu_k.
$$
Explain why $\mu_k$ can be interpreted as the center of cluster $k$.
Then compare the two conditional expectations
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
and
$$
\mathbb E[X_i\mid C_i=k].
$$
Explain why the first gives the soft cluster membership of an observed point, while the second gives the mean location of a cluster.

---

6. The Complete-Data Likelihood
If the latent labels $z_i$ were known, the complete-data likelihood would be
$$
p(x_1,\dots,x_n,z_1,\dots,z_n)=\prod_{i=1}^n
\prod_{k=1}^K
\left[
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$
Take the logarithm and show that the complete-data log-likelihood is
$$
\ell_c=\sum_{i=1}^n
\sum_{k=1}^K
z_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why this expression would be easy to maximize if the values of $z_{ik}$ were known.

---

7. The EM Interpretation:
In practice, the latent variables $Z_i$ are not observed. The EM algorithm replaces the unknown indicators $z_{ik}$ by their conditional expectations given the observed data and current parameter estimates:
$$
z_{ik}
\quad\leadsto\quad
\mathbb E[Z_{ik}\mid X_i=x_i].
$$
That is,
$$
z_{ik}
\quad\leadsto\quad
\gamma_{ik}.
$$
This is the E-step of the EM algorithm.
Using this idea, write the expected complete-data log-likelihood:
$$
Q=\sum_{i=1}^n
\sum_{k=1}^K
\gamma_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why the E-step can be interpreted as a conditional update of cluster membership probabilities.

---

8. Parameter Updates:
By maximizing $Q$ with respect to the model parameters, derive the standard GMM updates:
$$
N_k=\sum_{i=1}^n \gamma_{ik},
$$
$$
\phi_k^{\text{new}}=\frac{N_k}{n},
$$
$$
\mu_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}x_i,
$$
and
$$
\Sigma_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}
(x_i-\mu_k^{\text{new}})
(x_i-\mu_k^{\text{new}})^T.
$$
Explain how the responsibility $\gamma_{ik}$ acts as a fractional membership weight of observation $x_i$ in cluster $k$.

---

9. Interpretation:
Write a short paragraph explaining why GMM clustering can be viewed as a repeated process of conditional updating.
Your answer should mention the following points:

* The mixture weight $\phi_k$ is the prior probability of cluster $k$.
* The Gaussian density $\mathscr N(x_i\mid \mu_k,\Sigma_k)$ measures how compatible $x_i$ is with cluster $k$.
* The responsibility $\gamma_{ik}$ is the posterior probability of cluster $k$ after observing $x_i$.
* The soft assignment vector is
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

* The M-step updates the cluster parameters using these posterior membership probabilities as weights.
Conclude that Gaussian mixture clustering is probabilistic clustering based on conditional expectations of latent cluster membership variables.

---

Here is a perfectly tailored question that you can add as the final part (**Part 10**) of your assignment notebook to bridge your theoretical derivations with your code implementation:

---

10. Computational Simulation and Out-of-Sample Validation

Using the theoretical framework established in the previous parts, write a Python class named `GMMFinancialSegmenter` that implements a two-dimensional Gaussian Mixture Model (GMM) using `scikit-learn` and visualizes the results interactively using `Plotly`. Your implementation should fulfill the following criteria:

* **Data Splitting and Scaling:** Accept a dataset containing two continuous features (e.g., mimicking financial behaviors like `PURCHASES` and `CREDIT_LIMIT`), standardize the features to handle variance scaling, and split the data into an 80% training set and a 20% validation/test set.
* **EM Execution:** Fit a GMM with $K=3$ components on the training data using the Expectation-Maximization (EM) algorithm, printing whether the model successfully converged and the number of iterations required.
* **Out-of-Sample Performance:** Compute and output the average log-likelihood score over the unseen test set to validate how well the learned density functions generalize to new data.
* **Interactive Visualizations:** Implement methods to generate three distinct Plotly figures:
1. An empirical **2D Density Heatmap** of the raw training data with marginal distributions to inspect its underlying multimodal structure.
2. A **Training Assignment Plot** that overlays the training data points on top of a continuous contour map showing the maximum posterior responsibilities ($\gamma_{ik}$) computed across a fine coordinate grid.
3. A **Test Assignment Plot** that replicates the contour boundary visualization but overlays out-of-sample test data points to expose the physical regions of cluster ambiguity.



Briefly evaluate the resulting plots. Explain how the continuous background contour map visually demonstrates the soft assignment expectation vector $\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$ that you proved analytically in Part 3.

Use the dataset

https://www.kaggle.com/datasets/arjunbhasin2013/ccdata

1. Let $$(C_i)$$ denote the latent cluster assignment of observation $$(X_i)$$, where

$$[
P(C_i=k)=\phi_k,
\qquad
k=1,\ldots,K,
]$$

with

$$[
\phi_k\ge0,
\qquad
\sum_{k=1}^{K}\phi_k=1.
]$$

Conditional on the event

$$[
C_i=k,
]$$

the observation is assumed to follow a multivariate Gaussian distribution,

$$[
X_i\mid C_i=k
\sim
\mathcal N(\mu_k,\Sigma_k).
]$$

Using the Law of Total Probability, the marginal density of \(X_i\) is

$$[
p(x_i)
=
\sum_{k=1}^{K}
P(C_i=k)
p(x_i\mid C_i=k).
]$$

Substituting the prior probabilities and Gaussian densities gives

$$[
p(x_i)
=
\sum_{k=1}^{K}
\phi_k
\mathcal N(x_i\mid\mu_k,\Sigma_k).
]$$

Hence,

$$[
\boxed{
p(x_i)
=
\sum_{k=1}^{K}
\phi_k
\mathcal N(x_i\mid\mu_k,\Sigma_k)
}
]$$

This density is called a Gaussian mixture density because it is formed by combining several Gaussian probability density functions through a weighted sum. Each Gaussian component represents one cluster, while the mixture weights $$(\phi_k)$$ determine the probability that an observation originates from each cluster.

2. For an observed data point $$(x_i)$$, Bayes' theorem gives

$$[
P(C_i=k\mid X_i=x_i)
=
\frac{
P(X_i=x_i\mid C_i=k)
P(C_i=k)
}
{
P(X_i=x_i)
}.
]$$

Using the Law of Total Probability,

$$[
P(X_i=x_i)
=
\sum_{j=1}^{K}
P(X_i=x_i\mid C_i=j)
P(C_i=j).
]$$

Therefore,

$$[
P(C_i=k\mid X_i=x_i)
=
\frac{
P(X_i=x_i\mid C_i=k)
P(C_i=k)
}
{
\sum_{j=1}^{K}
P(X_i=x_i\mid C_i=j)
P(C_i=j)
}.
]$$

Substituting the Gaussian likelihood and mixture weights,

$$[
P(X_i=x_i\mid C_i=k)
=
\mathcal N(x_i\mid\mu_k,\Sigma_k),
]$$

and

$$[
P(C_i=k)
=
\phi_k,
]$$

gives

$$[
P(C_i=k\mid X_i=x_i)
=
\frac{
\phi_k
\mathcal N(x_i\mid\mu_k,\Sigma_k)
}
{
\sum_{j=1}^{K}
\phi_j
\mathcal N(x_i\mid\mu_j,\Sigma_j)
}.
]$$

Hence,

$$[
\boxed{
\gamma_{ik}
=
P(C_i=k\mid X_i=x_i)
=
\frac{
\phi_k
\mathcal N(x_i\mid\mu_k,\Sigma_k)
}
{
\sum_{j=1}^{K}
\phi_j
\mathcal N(x_i\mid\mu_j,\Sigma_j)
}
}
]$$

The quantity

$$[
\gamma_{ik}
]$$

is called the responsibility because it measures the proportion of responsibility that cluster $$(k)$$ takes for explaining observation $$(x_i)$$.

Since it satisfies

$$[
0\le\gamma_{ik}\le1,
]$$

and

$$[
\sum_{k=1}^{K}\gamma_{ik}=1,
]$$

it may be interpreted as the posterior probability that observation $$(x_i)$$ belongs to cluster $$(k)$$ after observing the data.

3. Define the one-hot encoded latent vector

$$[
Z_i=
\begin{bmatrix}
Z_{i1}\\
Z_{i2}\\
\vdots\\
Z_{iK}
\end{bmatrix},
]$$

where

$$[
Z_{ik}
=
\begin{cases}
1,
&
\text{if }
C_i=k,
\\
0,
&
\text{otherwise}.
\end{cases}
]$$

Since

$$[
Z_{ik}
]$$

is a Bernoulli random variable,

$$[
P(Z_{ik}=1\mid X_i=x_i)
=
P(C_i=k\mid X_i=x_i).
]$$

The conditional expectation of a Bernoulli random variable equals its probability of success. Therefore,

$$[
\mathbb E
[Z_{ik}\mid X_i=x_i]
=
1
\cdot
P(Z_{ik}=1\mid X_i=x_i)
+
0
\cdot
P(Z_{ik}=0\mid X_i=x_i).
]$$

Hence,

$$[
\boxed{
\mathbb E
[Z_{ik}\mid X_i=x_i]
=
P(C_i=k\mid X_i=x_i)
}
]$$

Substituting the posterior probability,

$$[
\boxed{
\mathbb E
[Z_{ik}\mid X_i=x_i]
=
\gamma_{ik}
}
]$$

Collecting all conditional expectations into a vector gives

$$[
\mathbb E
[Z_i\mid X_i=x_i]
=
\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}.
]$$

Therefore,

$$[
\boxed{
\mathbb E
[Z_i\mid X_i=x_i]
=
\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}
}
]$$

This vector represents the soft cluster assignment of observation $$(x_i)$$. Instead of assigning the observation completely to one cluster, the Gaussian Mixture Model assigns probabilities to all clusters according to their posterior probabilities.

Consequently, the soft cluster assignment in a Gaussian Mixture Model is precisely the conditional expectation

$$[
\boxed{
\mathbb E
[Z_i\mid X_i=x_i]
}
]$$

which provides the expected latent cluster membership after observing the data point.

 4.The conditional expectation

$$[
\mathbb{E}[Z_i \mid X_i=x_i]
=
\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}
]$$

represents the posterior probabilities that observation $$(x_i)$$ belongs to each of the $$(K)$$ clusters.

Since

$$[
0\le\gamma_{ik}\le1,
\qquad
\sum_{k=1}^{K}\gamma_{ik}=1,
]$$

each observation may simultaneously belong to several clusters with different probabilities.

A hard cluster assignment is obtained by assigning the observation to the cluster with the largest posterior probability,

$$[
\boxed{
\widehat{C}_i
=
\operatorname*{arg\,max}_{1\le k\le K}
\gamma_{ik}
}
]$$

In soft clustering, every observation contributes to every cluster according to its posterior probability. Thus, an observation located near the boundary between two clusters may have significant probabilities for both clusters.

In contrast, hard clustering assigns each observation exclusively to a single cluster by selecting only the largest posterior probability and discarding the remaining probabilities.

Therefore,

- **Soft clustering** represents uncertainty in cluster membership by assigning probabilities to all clusters.
- **Hard clustering** makes a single deterministic assignment based on the maximum posterior probability.

Gaussian Mixture Models naturally perform soft clustering, while hard clustering is obtained as a final decision rule by selecting the cluster with the highest responsibility.

5.Suppose that observation $$(X_i)$$ belongs to cluster $$(k)$$.

By assumption,

$$[
X_i
\mid
C_i=k
\sim
\mathcal N(\mu_k,\Sigma_k).
]$$

The expectation of a multivariate Gaussian distribution equals its mean vector. Therefore,

$$[
\boxed{
\mathbb E[X_i\mid C_i=k]
=
\mu_k
}
]$$

Hence, the parameter

$$[
\mu_k
]$$

represents the center of cluster $$(k)$$.

The two conditional expectations have different interpretations.

The conditional expectation

$$[
\mathbb E[Z_i\mid X_i=x_i]
=
\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}
]$$

describes the expected cluster membership of an observed data point. It provides the posterior probabilities that the observation belongs to each cluster and therefore represents a soft cluster assignment.

In contrast,

$$[
\mathbb E[X_i\mid C_i=k]
=
\mu_k
]$$

describes the expected location of all observations belonging to cluster $$(k)$$. It represents the geometric center of the cluster in the feature space.

Thus,

- $$(\mathbb E[Z_i\mid X_i=x_i])$$ provides information about **which cluster an observation belongs to**.
- $$(\mathbb E[X_i\mid C_i=k])$$ provides information about **where the cluster is located**.

The first expectation concerns the latent cluster membership variables, whereas the second concerns the observed feature vectors.


6. Suppose that the latent indicator variables

$$[
z_{ik}
]$$

are observed.

The complete-data likelihood is

$$[
p(x_1,\ldots,x_n,z_1,\ldots,z_n)
=
\prod_{i=1}^{n}
\prod_{k=1}^{K}
\left[
\phi_k
\mathcal N(x_i\mid\mu_k,\Sigma_k)
\right]^{z_{ik}}.
]$$

Taking the natural logarithm,

$$[
\ell_c
=
\log
p(x_1,\ldots,x_n,z_1,\ldots,z_n).
]$$

Using the logarithm of a product,

$$[
\log(ab)
=
\log a+\log b,
]$$

and

$$[
\log
\left(
a^b
\right)
=
b\log a,
]$$

gives

$$[
\ell_c
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
z_{ik}
\log
\left[
\phi_k
\mathcal N(x_i\mid\mu_k,\Sigma_k)
\right].
]$$

Applying the logarithm to the product,

$$[
\boxed{
\ell_c
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
z_{ik}
\left[
\log\phi_k
+
\log
\mathcal N(x_i\mid\mu_k,\Sigma_k)
\right]
}
]$$

If the values of

$$[
z_{ik}
]$$

were known, every observation would already be assigned to a cluster.

Consequently,

- the mixture weights could be estimated by counting observations assigned to each cluster,
- the cluster means would become weighted sample means,
- the covariance matrices would become weighted sample covariance matrices.

Thus, maximizing the complete-data log-likelihood becomes a straightforward optimization problem because the latent cluster assignments are no longer unknown.

7. In practice, the latent indicator variables

$$[
Z_i
]$$

are not observed.

Instead of using the unknown quantities

$$[
z_{ik},
]$$

the Expectation-Maximization (EM) algorithm replaces them by their conditional expectations given the observed data and the current parameter estimates,

$$[
\boxed{
z_{ik}
\longrightarrow
\mathbb E[Z_{ik}\mid X_i=x_i]
=
\gamma_{ik}
}
]$$

This replacement forms the Expectation Step (E-step).

Substituting the posterior responsibilities into the complete-data log-likelihood gives the expected complete-data log-likelihood,

$$[
Q
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
\gamma_{ik}
\left[
\log\phi_k
+
\log
\mathcal N(x_i\mid\mu_k,\Sigma_k)
\right].
]$$

Hence,

$$[
\boxed{
Q
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
\gamma_{ik}
\left[
\log\phi_k
+
\log
\mathcal N(x_i\mid\mu_k,\Sigma_k)
\right]
}
]$$

The E-step computes the posterior probability that each observation belongs to every cluster using the current parameter estimates.

Consequently, every observation contributes fractionally to each cluster rather than belonging entirely to a single cluster.

The E-step can therefore be interpreted as a conditional update of cluster membership probabilities because it computes

$$[
P(C_i=k\mid X_i=x_i),
]$$

which represents the posterior probability of cluster membership after observing the data.

8. The Maximization Step (M-step) updates the model parameters by maximizing the expected complete-data log-likelihood

$$[
Q
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
\gamma_{ik}
\left[
\log\phi_k
+
\log
\mathcal N(x_i\mid\mu_k,\Sigma_k)
\right].
]$$

The total effective membership of cluster $$(k)$$ is

$$[
\boxed{
N_k
=
\sum_{i=1}^{n}
\gamma_{ik}
}
]$$

Since the responsibilities satisfy

$$[
0\le\gamma_{ik}\le1,
]$$

the quantity $$(N_k)$$ represents the effective number of observations assigned to cluster $$(k)$$.



The mixture weights must satisfy

$$[
\sum_{k=1}^{K}\phi_k=1.
]$$

Using Lagrange multipliers to maximize $$(Q)$$ under this constraint gives

$$[
\boxed{
\phi_k^{\mathrm{new}}
=
\frac{N_k}{n}
}
]$$

where $$(n)$$ is the total number of observations.



Differentiating $$(Q)$$ with respect to the mean vector $$(\mu_k)$$ and setting the derivative equal to zero gives

$$[
\sum_{i=1}^{n}
\gamma_{ik}
(x_i-\mu_k)
=
0.
]$$

Rearranging,

$$[
\boxed{
\mu_k^{\mathrm{new}}
=
\frac{1}{N_k}
\sum_{i=1}^{n}
\gamma_{ik}x_i
}
]$$

Thus, each cluster mean is the weighted average of the observations, where the weights are the posterior responsibilities.



Differentiating $4(Q)$$ with respect to the covariance matrix gives

$$[
\boxed{
\Sigma_k^{\mathrm{new}}
=
\frac{1}{N_k}
\sum_{i=1}^{n}
\gamma_{ik}
(x_i-\mu_k^{\mathrm{new}})
(x_i-\mu_k^{\mathrm{new}})^T
}
]$$

The covariance matrix is therefore a weighted covariance of the observations assigned to cluster $$(k)$$.

The responsibility

$$[
\gamma_{ik}
=
P(C_i=k\mid X_i=x_i)
]$$

acts as a fractional membership weight. Instead of contributing completely to one cluster, each observation contributes proportionally to every cluster according to its posterior probability.

Consequently, observations lying near the boundary between clusters influence multiple clusters, while observations located near the center of a cluster contribute almost entirely to that cluster.

9. Gaussian Mixture Model clustering can be viewed as a repeated process of conditional updating. Initially, each cluster is assigned a prior probability

$$[
\phi_k,
]$$

which represents the probability that an observation belongs to cluster $$(k)$$ before the observation is seen.

After observing a data point $$(x_i)$$, the Gaussian density

$$[
\mathcal N(x_i\mid\mu_k,\Sigma_k)
]$$

measures how well the observation matches the characteristics of cluster $$(k)$$. Combining this likelihood with the prior probability through Bayes' theorem produces the posterior probability

$$[
\gamma_{ik}
=
P(C_i=k\mid X_i=x_i),
]$$

known as the responsibility.

The collection of posterior probabilities forms the soft assignment vector

$$[
\boxed{
\mathbb E[Z_i\mid X_i=x_i]
=
\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}
}
]$$

which gives the expected latent cluster membership of the observation.

During the Maximization Step, these posterior probabilities are treated as fractional membership weights for updating the mixture weights, cluster means, and covariance matrices.

The Expectation Step computes the posterior probabilities using the current parameter estimates, while the Maximization Step updates the parameters using those probabilities. These two steps are repeated until convergence.

Therefore, Gaussian Mixture Model clustering is fundamentally a probabilistic clustering method based on conditional expectations of latent cluster membership variables. Rather than assigning observations permanently to one cluster, the algorithm continuously updates both the cluster membership probabilities and the model parameters until a stable solution is reached.

In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture

import plotly.graph_objects as go
import plotly.express as px

In [4]:
from google.colab import files

uploaded = files.upload()

Saving CC GENERAL.csv to CC GENERAL.csv


In [5]:
import os

print(os.listdir())

['.config', 'CC GENERAL.csv', 'sample_data']


In [6]:
df = pd.read_csv("CC GENERAL.csv")

In [7]:
df.head()

,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,C10001,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,C10002,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,C10003,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,C10004,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,NaN,0.000000,12
4,C10005,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12


In [8]:
df = df[["PURCHASES","CREDIT_LIMIT"]]

In [9]:
df = df.dropna()

In [10]:
df.head()

,PURCHASES,CREDIT_LIMIT
0,95.40,1000.0
1,0.00,7000.0
2,773.17,7500.0
3,1499.00,7500.0
4,16.00,1200.0


In [11]:
X = df.values

In [12]:
X.shape

(8949, 2)

In [13]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [14]:
X_train, X_test = train_test_split(

    X_scaled,

    test_size=0.20,

    random_state=42

)

In [15]:
print(X_train.shape)

print(X_test.shape)

(7159, 2)
(1790, 2)


In [16]:
gmm = GaussianMixture(

    n_components=3,

    covariance_type="full",

    random_state=42

)

In [17]:
gmm.fit(X_train)

GaussianMixture(n_components=3, random_state=42)

In [18]:
print("Converged :", gmm.converged_)

print("Iterations :", gmm.n_iter_)

Converged : True
Iterations : 19


In [19]:
train_labels = gmm.predict(X_train)

In [20]:
test_labels = gmm.predict(X_test)

In [21]:
train_prob = gmm.predict_proba(X_train)

In [22]:
test_prob = gmm.predict_proba(X_test)

In [23]:
train_prob[:5]

array([[9.86510397e-01, 3.18079660e-08, 1.34895716e-02],
       [9.51674215e-01, 1.04589571e-11, 4.83257845e-02],
       [9.77465004e-01, 2.06381132e-03, 2.04711844e-02],
       [6.76241949e-01, 3.14123503e-01, 9.63454872e-03],
       [9.71166356e-01, 4.57719306e-10, 2.88336432e-02]])

In [24]:
score = gmm.score(X_test)

print(score)

-1.6464956236102442


In [25]:
fig = px.density_heatmap(

    x=X_train[:,0],

    y=X_train[:,1],

    nbinsx=40,

    nbinsy=40,

    marginal_x="histogram",

    marginal_y="histogram",

    title="Training Data Density"

)

fig.show()

In [26]:
x_min = X_train[:,0].min()-1

x_max = X_train[:,0].max()+1

y_min = X_train[:,1].min()-1

y_max = X_train[:,1].max()+1

In [27]:
xx,yy = np.meshgrid(

np.linspace(x_min,x_max,300),

np.linspace(y_min,y_max,300)

)

In [28]:
grid = np.column_stack(

(xx.ravel(),

yy.ravel())

)

In [29]:
prob = gmm.predict_proba(grid)

In [30]:
Z = np.max(prob,axis=1)

In [31]:
Z = Z.reshape(xx.shape)

In [32]:
fig = go.Figure()

In [33]:
fig.add_trace(

go.Contour(

x=xx[0],

y=yy[:,0],

z=Z,

colorscale="Viridis",

opacity=0.6

)

)

In [34]:
fig.add_trace(

go.Scatter(

x=X_train[:,0],

y=X_train[:,1],

mode="markers",

marker=dict(

color=train_labels,

size=6

),

name="Training"

)

)

In [35]:
fig.show()

In [36]:
fig = go.Figure()

In [37]:
fig.add_trace(

go.Scatter(

x=X_test[:,0],

y=X_test[:,1],

mode="markers",

marker=dict(

color=test_labels,

size=7

),

name="Testing"

)

)

In [38]:
fig.show()